# Agregaciones y Agrupaciones

Iniciar Sesión de Spark (Spark Session)

---

In [ ]:
from pyspark.sql import SparkSession

# crear la sesión
spark = SparkSession \
        .builder \
        .appName("DataFrames Basics") \
        .master("local[*]") \
        .getOrCreate()

spark.version

In [ ]:
spark

In [ ]:
# Para optimización de conversión a Pandas
spark.conf.set("spark.sql.execution.arrow.enabled", "true")

In [ ]:
# Importar funciones sql
from pyspark.sql.functions import *

In [ ]:
planetasDF = spark.read \
    .option("inferSchema", True) \
    .option("header", "true") \
    .option("delimiter", ";") \
    .csv("planets.csv")

planetasDF = planetasDF \
    .withColumn("population",
                when(col("population").rlike("^[0-9]+$"), col("population")).otherwise(None).cast("int")) \
    .withColumn("diameter",
                when(col("diameter").rlike("^[0-9]+$"), col("diameter")).otherwise(None).cast("int"))

In [ ]:
planetasDF.show(2, False)
print(planetasDF.schema.fields)
planetasDF.columns

## Examples

Count

In [ ]:
# Recuento de filas df, incluyendo NULLS
planetasDF.count()

In [ ]:
# utilizando funciones SQL, SIN incluir NULLS
climasCountDF = planetasDF.select(count(col("climate")))
climasCountDF.show()

In [ ]:
terrenosCountDF = planetasDF.select(count(planetasDF.terrain))
terrenosCountDF.show()

In [ ]:
planetasDF.select(count(planetasDF.climate).alias("countClimas"), count(planetasDF.terrain)).show()

In [ ]:
#usando sintaxis SQL
planetasDF.select(expr("count(terrain)")).show()
planetasDF.selectExpr("count(terrain) as count").show()

In [ ]:
#usando SQL (creando una tabla temporal)
planetasDF.createOrReplaceTempView("planetas")

In [ ]:
spark.sql("select count(terrain) from planetas").show()

In [ ]:
spark.sql("select count(terrain) as countTerrenos, count(climate) from planetas").show()

Count Distinct

In [ ]:
planetasDF.select(countDistinct(planetasDF.climate)).show()

In [ ]:
spark.sql("select count(distinct climate) from planetas").show()

Min y max

In [ ]:
planetasDF.select(min(planetasDF.population), max(planetasDF.population)).show()

In [ ]:
spark.sql("select min(population) from planetas").show()

Sum

In [ ]:
planetasDF.select(sum(planetasDF.orbital_period).alias("suma_periodo_orbital")).show()
planetasDF.selectExpr("sum(orbital_period) as periodo_orbital_total").show()

Average (promedio)

In [ ]:
planetasDF.select(avg(planetasDF.diameter)).show()
spark.sql("select avg(diameter) from planetas").show()

Estadísticos

In [ ]:
planetasDF.select(mean(planetasDF.population)).show()
planetasDF.select(stddev(planetasDF.population)).show()

### Agrupaciones

---

In [ ]:
countByClimaDF = planetasDF.groupBy(planetasDF.climate).count().orderBy("count")
countByClimaDF.show()

In [ ]:
spark.sql("select climate, count(climate) as count from planetas where climate is not null group by climate order by count").show()

In [ ]:
avgDiametroByClimaDF = planetasDF.groupBy(col("climate")).avg("diameter").orderBy(col("avg(diameter)").desc())
avgDiametroByClimaDF.show()

In [ ]:
planetasDF.groupBy(col("climate")).agg(avg("diameter") \
    .alias("avg")).orderBy(col("avg").desc()).show()

In [ ]:
aggregationsByGenreDF = planetasDF.groupBy("climate") \
    .agg(
        count("*").alias("Numero_de_planetas"),
        avg("diameter").alias("promedio")
    ) \
    .orderBy(col("promedio").desc()).show()

## Ejercicios

   1. Suma todos los cost_in_credits de TODOS los vehículos del archivo vehicles.csv. A continuación, suma los cost_in_credits por vehicle_class.
   
   2. Cuenta cuántos manufacturer distintos tenemos.
   
   3. Muestra la media y la desviación estándar de los passengers (de todos los vehículos) y después solo de los vehicle_class (repulsorcraft).
   
   4. Calcula el max_atmosphering_speed medio y la length media POR vehicle_class.
   
   5. Suma TODAS las cargo_capacity de TODOS los vehciulos en el DF. A continuación, suma TODOS los valores de crew por vehicle_class. ¿Ve valores nulos? ¿Por qué? ¿Cómo puede resolverlo?

Ejercicio 1

Ejercicio 2

Ejercicio 3

Ejercicio 4

Ejercicio 5